# Preprocessing 모듈 테스트

이 노트북은 `preprocessing` 모듈, 특히 `format_dataset.py` 스크립트의 기능을 테스트합니다.
1. 필요한 라이브러리와 모듈을 임포트합니다.
2. 테스트용 입력 오디오 및 텍스트 파일을 생성합니다 (TTS 출력 시뮬레이션).
3. `format_dataset.py`의 메타데이터 생성 함수를 실행합니다.
4. 생성된 메타데이터 파일 (`metadata.csv`)을 확인합니다.

*참고: `clean_audio.py`의 실제 기능은 구현되지 않았으므로, 해당 부분 테스트는 포함되지 않습니다.*

In [2]:
import os
import sys
import pandas as pd
import glob

# 프로젝트 루트를 sys.path에 추가하여 모듈 임포트
project_root = os.path.abspath('.') # 노트북이 프로젝트 루트에 있다고 가정
if project_root not in sys.path:
    sys.path.append(project_root)

from preprocessing.format_dataset import create_metadata_file
from preprocessing.clean_audio import process_audio_directory # 필요시 주석 해제
from utils.file_utils import ensure_dir_exists
from utils.logging_utils import setup_logger

logger = setup_logger('preprocessing_test_notebook')

## 1. 설정 및 경로 정의

In [3]:
test_audio_dir = 'data/test_preprocessing/audio' # 테스트용 오디오 디렉토리
test_text_dir = 'data/test_preprocessing/text'   # 테스트용 텍스트 디렉토리
output_metadata_dir = 'data/test_preprocessing/dataset'
output_metadata_filename = 'test_metadata.csv'
output_metadata_path = os.path.join(output_metadata_dir, output_metadata_filename)

# 필요한 디렉토리 생성
ensure_dir_exists(test_audio_dir)
ensure_dir_exists(test_text_dir)
ensure_dir_exists(output_metadata_dir)

print(f"Test Audio Directory: {test_audio_dir}")
print(f"Test Text Directory: {test_text_dir}")
print(f"Output Metadata Path: {output_metadata_path}")

Created directory: data/test_preprocessing/audio
Created directory: data/test_preprocessing/text
Created directory: data/test_preprocessing/dataset
Test Audio Directory: data/test_preprocessing/audio
Test Text Directory: data/test_preprocessing/text
Output Metadata Path: data/test_preprocessing/dataset/test_metadata.csv


## 2. 테스트용 입력 파일 생성 (TTS 출력 시뮬레이션)

`format_dataset.py`는 오디오 파일과 동일한 이름의 `.txt` 파일이 있다고 가정합니다.

In [4]:
sample_data = {
    "sample_01": "첫 번째 샘플 오디오입니다.",
    "sample_02": "두 번째 오디오 파일에 대한 텍스트입니다.",
    "sample_03": "세 번째 문장입니다."
}

audio_format = '.wav' # 테스트용 오디오 형식 (실제 파일 생성 안 함)

for base_name, text in sample_data.items():
    # 텍스트 파일 생성
    text_path = os.path.join(test_text_dir, f"{base_name}.txt")
    try:
        with open(text_path, 'w', encoding='utf-8') as f:
            f.write(text)
        logger.info(f"Created sample text file: {text_path}")
    except Exception as e:
        logger.error(f"Failed to create sample text file {text_path}: {e}")
        
    # 빈 오디오 파일 생성 (내용은 중요하지 않음, 파일 존재 여부만 확인)
    audio_path = os.path.join(test_audio_dir, f"{base_name}{audio_format}")
    try:
        with open(audio_path, 'w') as f:
            pass # Create an empty file
        logger.info(f"Created dummy audio file: {audio_path}")
    except Exception as e:
        logger.error(f"Failed to create dummy audio file {audio_path}: {e}")
        
# 매칭되지 않는 오디오 파일 추가 (텍스트 없음)
unmatched_audio_path = os.path.join(test_audio_dir, f"unmatched_sample{audio_format}")
try:
    with open(unmatched_audio_path, 'w') as f:
        pass
    logger.info(f"Created dummy unmatched audio file: {unmatched_audio_path}")
except Exception as e:
    logger.error(f"Failed to create dummy unmatched audio file: {e}")

2025-04-16 08:25:01,038 - preprocessing_test_notebook - INFO - Created sample text file: data/test_preprocessing/text/sample_01.txt
2025-04-16 08:25:01,040 - preprocessing_test_notebook - INFO - Created dummy audio file: data/test_preprocessing/audio/sample_01.wav
2025-04-16 08:25:01,041 - preprocessing_test_notebook - INFO - Created sample text file: data/test_preprocessing/text/sample_02.txt
2025-04-16 08:25:01,042 - preprocessing_test_notebook - INFO - Created dummy audio file: data/test_preprocessing/audio/sample_02.wav
2025-04-16 08:25:01,043 - preprocessing_test_notebook - INFO - Created sample text file: data/test_preprocessing/text/sample_03.txt
2025-04-16 08:25:01,044 - preprocessing_test_notebook - INFO - Created dummy audio file: data/test_preprocessing/audio/sample_03.wav
2025-04-16 08:25:01,045 - preprocessing_test_notebook - INFO - Created dummy unmatched audio file: data/test_preprocessing/audio/unmatched_sample.wav
2025-04-16 08:25:01,040 - preprocessing_test_notebook -

## 3. 메타데이터 생성 실행 (`format_dataset.py`)

In [5]:
logger.info(f"Running create_metadata_file for audio dir '{test_audio_dir}' and text dir '{test_text_dir}'")
success = create_metadata_file(test_audio_dir, test_text_dir, output_metadata_path)

if success:
    logger.info("Metadata file creation process completed.")
else:
    logger.error("Metadata file creation process failed.")

2025-04-16 08:25:19,865 - preprocessing_test_notebook - INFO - Running create_metadata_file for audio dir 'data/test_preprocessing/audio' and text dir 'data/test_preprocessing/text'
2025-04-16 08:25:19,868 - preprocessing.format_dataset - INFO - Creating metadata file for audio in 'data/test_preprocessing/audio' and text in 'data/test_preprocessing/text'
2025-04-16 08:25:19,871 - preprocessing.format_dataset - WARNING - Transcription file not found for audio: unmatched_sample.wav. Expected at: data/test_preprocessing/text/unmatched_sample.txt
2025-04-16 08:25:19,868 - preprocessing.format_dataset - INFO - Creating metadata file for audio in 'data/test_preprocessing/audio' and text in 'data/test_preprocessing/text'
2025-04-16 08:25:19,871 - preprocessing.format_dataset - WARNING - Transcription file not found for audio: unmatched_sample.wav. Expected at: data/test_preprocessing/text/unmatched_sample.txt
2025-04-16 08:25:19,880 - preprocessing.format_dataset - INFO - Metadata file create

## 4. 생성된 메타데이터 파일 확인

In [6]:
if os.path.exists(output_metadata_path):
    logger.info(f"Metadata file found at: {output_metadata_path}")
    try:
        df = pd.read_csv(output_metadata_path)
        print("\nMetadata file content:")
        display(df)
        
        # 간단한 검증
        expected_rows = len(sample_data)
        if len(df) == expected_rows:
            logger.info(f"Metadata file contains the expected number of rows ({expected_rows}).")
        else:
            logger.warning(f"Metadata file contains {len(df)} rows, but expected {expected_rows}.")
            
        # 오디오 파일 경로가 절대 경로인지 확인 (create_metadata_file 구현에 따라 다름)
        if not df.empty and os.path.isabs(df.iloc[0]['audio_filepath']):
             logger.info("Audio file paths seem to be absolute.")
        elif not df.empty:
             logger.warning("Audio file paths might not be absolute. Check implementation if absolute paths are needed.")
            
    except Exception as e:
        logger.error(f"Failed to read or display the metadata file: {e}")
else:
    logger.error(f"Metadata file was not created at {output_metadata_path}.")

2025-04-16 08:25:44,873 - preprocessing_test_notebook - INFO - Metadata file found at: data/test_preprocessing/dataset/test_metadata.csv

Metadata file content:

Metadata file content:


,audio_filepath,text
0,/workspaces/Luciper/data/test_preprocessing/au...,두 번째 오디오 파일에 대한 텍스트입니다.
1,/workspaces/Luciper/data/test_preprocessing/au...,첫 번째 샘플 오디오입니다.
2,/workspaces/Luciper/data/test_preprocessing/au...,세 번째 문장입니다.


2025-04-16 08:25:44,891 - preprocessing_test_notebook - INFO - Metadata file contains the expected number of rows (3).
2025-04-16 08:25:44,892 - preprocessing_test_notebook - INFO - Audio file paths seem to be absolute.
2025-04-16 08:25:44,892 - preprocessing_test_notebook - INFO - Audio file paths seem to be absolute.


## 5. 오디오 클리닝 테스트 (`clean_audio.py`)

TTS 테스트에서 생성된 오디오 파일이나 다른 샘플 오디오 파일을 사용하여 `process_audio_directory` 함수를 테스트합니다.

In [7]:
import soundfile as sf
import numpy as np

# --- Setup for clean_audio test ---
clean_input_dir = 'data/test_preprocessing/audio_raw' # Use a different input dir for cleaning test
clean_output_dir = 'data/test_preprocessing/audio_cleaned'

ensure_dir_exists(clean_input_dir)
ensure_dir_exists(clean_output_dir)

# Create some dummy audio files for testing clean_audio
sr = 22050 # Create with a different sample rate
duration = 3
frequency = 440
t = np.linspace(0., duration, int(sr * duration))
# File 1: Normal tone with silence
amplitude = 0.5
silence_len = int(sr * 0.5)
audio1 = np.concatenate([np.zeros(silence_len), amplitude * np.sin(2. * np.pi * frequency * t), np.zeros(silence_len)])
sf.write(os.path.join(clean_input_dir, 'test_audio_01.wav'), audio1, sr)

# File 2: Low volume tone
amplitude_low = 0.05
audio2 = amplitude_low * np.sin(2. * np.pi * frequency * 1.5 * t)
sf.write(os.path.join(clean_input_dir, 'test_audio_02.wav'), audio2, sr)

# File 3: Silent file
audio3 = np.zeros(int(sr * 2))
sf.write(os.path.join(clean_input_dir, 'test_audio_03.wav'), audio3, sr)

logger.info(f"Created dummy audio files in {clean_input_dir}")

# --- Run the cleaning process ---
from preprocessing.clean_audio import process_audio_directory

logger.info(f"Running process_audio_directory on {clean_input_dir} -> {clean_output_dir}")
process_audio_directory(clean_input_dir, clean_output_dir, normalize=True, trim_silence=True)

# --- Verify the output ---
cleaned_files = glob.glob(os.path.join(clean_output_dir, '*.wav'))
logger.info(f"Found {len(cleaned_files)} cleaned files in {clean_output_dir}:")
for f_path in cleaned_files:
    try:
        info = sf.info(f_path)
        logger.info(f" - {os.path.basename(f_path)}: SR={info.samplerate}, Channels={info.channels}, Duration={info.duration:.2f}s")
        # Add more checks if needed (e.g., check RMS level, check if silence is trimmed)
    except Exception as e:
        logger.error(f"Could not get info for {f_path}: {e}")

Created directory: data/test_preprocessing/audio_raw
Created directory: data/test_preprocessing/audio_cleaned
2025-04-16 08:30:43,493 - preprocessing_test_notebook - INFO - Created dummy audio files in data/test_preprocessing/audio_raw
2025-04-16 08:30:43,495 - preprocessing_test_notebook - INFO - Running process_audio_directory on data/test_preprocessing/audio_raw -> data/test_preprocessing/audio_cleaned
2025-04-16 08:30:43,495 - preprocessing.clean_audio - INFO - Starting audio cleaning process for directory: data/test_preprocessing/audio_raw
2025-04-16 08:30:43,496 - preprocessing.clean_audio - INFO - Normalizing volume for data/test_preprocessing/audio_raw/test_audio_02.wav to data/test_preprocessing/audio_cleaned/test_audio_02.wav
2025-04-16 08:30:43,497 - preprocessing.clean_audio - INFO - Volume normalization (placeholder) complete for data/test_preprocessing/audio_cleaned/test_audio_02.wav
2025-04-16 08:30:43,497 - preprocessing.clean_audio - INFO - Placeholder processing compl